In [1]:
!pip install -q transformers torch sentencepiece sentence-transformers nltk pandas requests

In [23]:
import json
import re
import requests
import torch
import nltk
import pandas as pd

from nltk.tokenize import sent_tokenize
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification
)
from sentence_transformers import SentenceTransformer, util

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Running on:", device)

Running on: cuda


In [24]:
url = "https://raw.githubusercontent.com/salesforce/DiverseSumm/main/data/diverse_summ.json"

response = requests.get(url, timeout=120)
response.raise_for_status()

data = response.json()

print("DiverseSumm loaded successfully!")
print("Number of events:", len(data))

DiverseSumm loaded successfully!
Number of events: 245


In [25]:
target_url = "https://www.usatoday.com/story/news/politics/2023/03/20/trump-desantis-bragg-stormy-daniels-money/11509124002/"

def find_article(obj):
    if isinstance(obj, dict):
        if str(obj.get("url", "")).rstrip("/") == target_url.rstrip("/"):
            return obj

        for value in obj.values():
            result = find_article(value)
            if result is not None:
                return result

    elif isinstance(obj, list):
        for item in obj:
            result = find_article(item)
            if result is not None:
                return result

    return None

article = find_article(data)

if article is None:
    raise ValueError("Article not found in DiverseSumm")

article_text = article["content"]

print("Article found!")
print("Characters:", len(article_text))
print("\nPreview:\n")
print(article_text[:1000])

Article found!
Characters: 3834

Preview:

Ron DeSantis sought to have it all sorts of ways Monday on Donald Trump's possible indictment, criticizing the prosecutor in charge of the case but also pointing out that it involves "porn star hush money payments."

The comments drew political attacks from Trump and his allies, who are trying to pressure Republicans for support at a time when he faces the prospect of being the first former president in history to be indicted.

At a news conference in Florida, DeSantis said he would not get involved in the case but did criticize Manhattan District Attorney Alvin Bragg for considering prosecution of a former president.

Bragg "chooses to go back many, many years ago, to try to use something about porn star hush money payments," DeSantis said. "You know, that's an example of pursuing a political agenda and weaponizing the office."

Trump aides interpreted the "porn star" reference as an attack and described his defense as less than enthusiastic.

In [7]:
# Start with the first 500 words of the article
passage = " ".join(article_text.split()[:500])

# Tokenize the passage
inputs = tokenizer(
    passage,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(device)

# Generate claims
model.eval()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        num_beams=4
    )

# Decode output
claims = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("FENICE output:\n")
print(claims)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FENICE output:

Ron DeSantis sought to have it all sorts of ways on Donald Trump's possible indictment. He criticized the prosecutor in charge of the case. He pointed out that the case involves 'porn star hush money payments'. The comments drew political attacks from Trump and his allies. Trump and his allies are trying to pressure Republicans for support. Trump faces the prospect of being the first former president in history to be indicted. Trump aides interpreted the 'porn star' reference as an attack. Trump can't even say his name! Trump aides interpreted the 'porn star' reference as an attack. Trump aides described his defense as less than enthusiastic. Trump aides cited unverified accusations about DeSantis during his high school teaching days. Trump cited unverified accusations about DeSantis during his high school teaching days. Trump cited unverified accusations about DeSantis during his high school teaching days. Trump cited a threat to the Florida governor. Trump cited unver

In [26]:
fenice_name = "Babelscape/t5-base-summarization-claim-extractor"

fenice_tokenizer = AutoTokenizer.from_pretrained(fenice_name)

fenice_model = AutoModelForSeq2SeqLM.from_pretrained(
    fenice_name
).to(device)

fenice_model.eval()

print("FENICE loaded successfully!")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

FENICE loaded successfully!


In [27]:
sentences = sent_tokenize(article_text)

chunks = []
current_chunk = []

for sentence in sentences:
    candidate = " ".join(current_chunk + [sentence])

    if (
        len(fenice_tokenizer.encode(candidate)) > 300
        and current_chunk
    ):
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentence]
    else:
        current_chunk.append(sentence)

if current_chunk:
    chunks.append(" ".join(current_chunk))

# Guard against unusually long individual sentences
if any(len(fenice_tokenizer.encode(c)) > 512 for c in chunks):
    raise ValueError("A chunk exceeds 512 tokens. Split long sentences.")

print("Total sentences:", len(sentences))
print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks, 1):
    print(
        f"Chunk {i}: "
        f"{len(fenice_tokenizer.encode(chunk))} tokens"
    )

Total sentences: 37
Total chunks: 4
Chunk 1: 283 tokens
Chunk 2: 268 tokens
Chunk 3: 296 tokens
Chunk 4: 64 tokens


In [28]:
all_claims = []

for i, chunk in enumerate(chunks):

    inputs = fenice_tokenizer(
        chunk,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = fenice_model.generate(
            **inputs,
            max_new_tokens=384,
            num_beams=4,
            repetition_penalty=1.2
        )

    result = fenice_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    all_claims.append(result)

    print(f"\n{'='*50}")
    print(f"CHUNK {i+1}")
    print(f"{'='*50}")
    print(result)

[transformers] Both `max_new_tokens` (=384) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=384) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CHUNK 1
Ron DeSantis sought to have it all sorts of ways Monday on Donald Trump's possible indictment. The comments drew political attacks from Trump and his allies. Trump and his allies are trying to pressure Republicans for support at a time when he faces the prospect of being the first former president in history to be indicted. DeSantis said he would not get involved in the case. He did criticize Manhattan District Attorney Alvin Bragg for considering prosecution of a former president. Bragg chooses to go back many, many years ago, to try to use something about porn star hush money payments. Trump aides interpreted the 'porn star' reference as an attack. Trump spokeswoman Liz Harrington described his defense as less than enthusiastic.


[transformers] Both `max_new_tokens` (=384) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CHUNK 2
Trump cited unverified accusations about DeSantis during his high school teaching days. Trump issued a threat to the Florida governor. Ron DeSantis will probably find out about FALSE ACCUSATIONS & FAKE STORIES sometime in the future. Trump said on his Truth Social platform that he expected to be arrested Tuesday. The case revolves around payments to Stormy Daniels. There's no indication anything will happen Tuesday. The grand jury in New York heard Monday from Robert Costello, a Republican lawyer with ties to Trump's legal team. Trump and supporters demanded statements of solidarity from other Republicans. DeSantis was a prominent target of these demands.


[transformers] Both `max_new_tokens` (=384) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CHUNK 3
The Florida governor responded to a reporter's question about whether he would get involved in the potential extradition of Trump to New York City. DeSantis replied, 'We are not involved in this, won't be involved in this.' The Trump reaction quickly followed. DeSantis thinks that Dems weaponizing the law to indict President Trump is a'manufactured circus' & isn't a'real issue.' Former Vice President Mike Pence criticized Bragg's prosecution. Former South Carolina Gov. Nikki Haley has not commented on Trump's possible indictment in the New York case. Trump faces possible legal charges in Atlanta and Washington, D.C., over efforts to overturn his loss in the 2020 election and the insurrection of Jan. 6, 2021.

CHUNK 4
A Justice Department special counsel is investigating Trump over the handling of classified material. All the candidates noted that the prospect of criminal charges remains, at this point, speculative. DeSantis said, 'So I've seen rumors swirl.' DeSantis said, 'I 

In [29]:
extracted_claims = []

for output in all_claims:

    # Protect common title abbreviations during splitting
    cleaned = re.sub(
        r"\bGov\.",
        "Governor",
        output
    )

    claims = sent_tokenize(cleaned)

    for claim in claims:
        claim = claim.strip()

        if claim:
            extracted_claims.append(claim)

print("Total extracted claims:", len(extracted_claims))

for i, claim in enumerate(extracted_claims, 1):
    print(f"{i}. {claim}")

Total extracted claims: 28
1. Ron DeSantis sought to have it all sorts of ways Monday on Donald Trump's possible indictment.
2. The comments drew political attacks from Trump and his allies.
3. Trump and his allies are trying to pressure Republicans for support at a time when he faces the prospect of being the first former president in history to be indicted.
4. DeSantis said he would not get involved in the case.
5. He did criticize Manhattan District Attorney Alvin Bragg for considering prosecution of a former president.
6. Bragg chooses to go back many, many years ago, to try to use something about porn star hush money payments.
7. Trump aides interpreted the 'porn star' reference as an attack.
8. Trump spokeswoman Liz Harrington described his defense as less than enthusiastic.
9. Trump cited unverified accusations about DeSantis during his high school teaching days.
10. Trump issued a threat to the Florida governor.
11. Ron DeSantis will probably find out about FALSE ACCUSATIONS & 

In [30]:
sbert = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device=device
)

print("SBERT loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT loaded successfully!


In [31]:
# Encode original article sentences
sentence_embeddings = sbert.encode(
    sentences,
    convert_to_tensor=True,
    normalize_embeddings=True
)

# Encode extracted claims
claim_embeddings = sbert.encode(
    extracted_claims,
    convert_to_tensor=True,
    normalize_embeddings=True
)

# Calculate cosine similarity
similarities = util.cos_sim(
    claim_embeddings,
    sentence_embeddings
)

claim_evidence = []
claim_evidence_indices = []

for i, claim in enumerate(extracted_claims):

    top_indices = torch.topk(
        similarities[i],
        k=min(3, len(sentences))
    ).indices.tolist()

    evidence = [sentences[j] for j in top_indices]

    claim_evidence.append(evidence)
    claim_evidence_indices.append(top_indices)

# Preview first 3 claims
for i in range(min(3, len(extracted_claims))):
    print(f"\nCLAIM {i+1}: {extracted_claims[i]}")

    for j, evidence in enumerate(claim_evidence[i], 1):
        print(f"  Evidence {j}: {evidence}")


CLAIM 1: Ron DeSantis sought to have it all sorts of ways Monday on Donald Trump's possible indictment.
  Evidence 1: Ron DeSantis sought to have it all sorts of ways Monday on Donald Trump's possible indictment, criticizing the prosecutor in charge of the case but also pointing out that it involves "porn star hush money payments."
  Evidence 2: Trump himself weighed in by citing unverified accusations about DeSantis during his high school teaching days - and issuing a threat to the Florida governor.
  Evidence 3: DeSantis was a prominent target of these demands.

CLAIM 2: The comments drew political attacks from Trump and his allies.
  Evidence 1: The comments drew political attacks from Trump and his allies, who are trying to pressure Republicans for support at a time when he faces the prospect of being the first former president in history to be indicted.
  Evidence 2: Trump aides interpreted the "porn star" reference as an attack and described his defense as less than enthusiastic

In [32]:
nli_name = "cross-encoder/nli-deberta-v3-base"

nli_tokenizer = AutoTokenizer.from_pretrained(nli_name)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    nli_name
).to(device)

nli_model.eval()

print("NLI model loaded successfully!")
print("Labels:", nli_model.config.id2label)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

NLI model loaded successfully!
Labels: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}


In [33]:
THRESHOLD = 0.8

entailment_id = next(
    i for i, label in nli_model.config.id2label.items()
    if label.lower() == "entailment"
)

def evaluate_evidence(evidence_list, claim):

    inputs = nli_tokenizer(
        evidence_list,
        [claim] * len(evidence_list),
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        logits = nli_model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)

    scores = probabilities[:, entailment_id]

    best_idx = torch.argmax(scores).item()

    return (
        scores[best_idx].item(),
        evidence_list[best_idx]
    )


faithfulness_results = []

for i, claim in enumerate(extracted_claims):

    # First: evaluate individual sentences
    sentence_score, sentence_evidence = evaluate_evidence(
        claim_evidence[i],
        claim
    )

    best_score = sentence_score
    best_evidence = sentence_evidence
    evidence_type = "Single sentence"

    # Second: use context only when needed
    if sentence_score < THRESHOLD:

        windows = []

        for idx in claim_evidence_indices[i]:
            start = max(0, idx - 1)
            end = min(len(sentences), idx + 2)

            windows.append(" ".join(sentences[start:end]))

        context_score, context_evidence = evaluate_evidence(
            windows,
            claim
        )

        if context_score > best_score:
            best_score = context_score
            best_evidence = context_evidence
            evidence_type = "Context window"

    faithfulness_results.append({
        "Claim ID": i + 1,
        "Claim": claim,
        "Entailment Score": round(best_score, 4),
        "Evidence Type": evidence_type,
        "Best Evidence": best_evidence,
        "Faithfulness": (
            "SUPPORTED"
            if best_score >= THRESHOLD
            else "NEEDS REVIEW"
        )
    })

faithfulness_df = pd.DataFrame(faithfulness_results)

print(
    faithfulness_df[
        [
            "Claim ID",
            "Claim",
            "Entailment Score",
            "Evidence Type",
            "Faithfulness"
        ]
    ].to_string(index=False)
)

print("\nSummary:")
print(faithfulness_df["Faithfulness"].value_counts())

support_rate = (
    faithfulness_df["Faithfulness"].eq("SUPPORTED").mean()
)

print(f"\nAutomated support rate: {support_rate:.2%}")

 Claim ID                                                                                                                                                                   Claim  Entailment Score   Evidence Type Faithfulness
        1                                                                          Ron DeSantis sought to have it all sorts of ways Monday on Donald Trump's possible indictment.            0.9956 Single sentence    SUPPORTED
        2                                                                                                          The comments drew political attacks from Trump and his allies.            0.9976 Single sentence    SUPPORTED
        3 Trump and his allies are trying to pressure Republicans for support at a time when he faces the prospect of being the first former president in history to be indicted.            0.9936 Single sentence    SUPPORTED
        4                                                                                           

In [34]:
output_file = "fenice_sbert_faithfulness.csv"

faithfulness_df.to_csv(
    output_file,
    index=False
)

from google.colab import files
files.download(output_file)

print("Results saved successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Results saved successfully!
